# abbasdeep — استوديو الأصوات

هذا الدفتر هو نقطة التشغيل الفعلية لمشروع **abbasdeep**. يجهّز محرك Applio داخل جلسة Google Colab، يحفظ ملفات النماذج في Google Drive، ثم يعرض واجهة abbasdeep مباشرة داخل مخرجات الدفتر.

**لا تحتاج إلى نسخ رابط Gradio أو لصقه في موقع آخر.** ستظهر الواجهة داخل خلية التشغيل بعد الإعداد.

### طريقة الاستخدام
1. افتح الدفتر بحساب Google الخاص بك.
2. شغّل خلية ربط Google Drive ووافق على الصلاحية المطلوبة.
3. شغّل خلية تثبيت وتجهيز المحرك وانتظر اكتمالها.
4. شغّل خلية مزامنة الملفات أول مرة، ثم خلية تشغيل abbasdeep.
5. استخدم واجهة abbasdeep التي تظهر داخل الدفتر.

[مستودع المشروع](https://github.com/dharymax/abbasdeep) · [محرك Applio](https://github.com/IAHispano/Applio)

## تنبيه الاستخدام والخصوصية

- استخدم أصواتك أو أصواتاً لديك موافقة صريحة على معالجتها.
- لا تستخدم التطبيق لانتحال شخصية شخص حقيقي أو لتضليل المستمعين.
- لا تشارك رابط Gradio العام الناتج من الجلسة؛ من يمتلكه قد يصل إلى واجهة التشغيل ما دامت الجلسة فعالة.
- موارد GPU وذاكرة Colab ومدة الجلسة متغيرة وغير مضمونة.
- سياسة Google Colab تحظر إنشاء التزييف العميق، وقد تنهي الجلسات المجانية عند محاولة تجاوز واجهة الدفتر إلى واجهة ويب خارجية لمعالجة المحتوى. لذلك تعرض واجهة abbasdeep داخل مخرجات الدفتر ولا تعتمد على موقع خارجي لقيادة الجلسة.

[سياسة Colab الرسمية](https://research.google.com/colaboratory/faq.html)

In [ ]:
# @title التحقق من توفر GPU
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    print("⚠️ لم يظهر nvidia-smi في هذه الجلسة؛ يبدو أن Colab متصل بـ CPU أو لم يخصص GPU.")
    print("افتح Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.")
    print("بعد تغيير النوع، أعد الاتصال بالجلسة ثم شغّل خلية الفحص مرة أخرى.")
else:
    result = subprocess.run(
        [nvidia_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode == 0 and result.stdout.strip():
        print("✅ تم تخصيص GPU لهذه الجلسة:")
        print(result.stdout.strip())
    else:
        print("⚠️ الأمر nvidia-smi موجود لكن لم يتمكن من عرض معلومات GPU.")
        if result.stderr.strip():
            print(result.stderr.strip())

try:
    import torch
    if torch.cuda.is_available():
        print(f"✅ PyTorch يرى CUDA: {torch.cuda.get_device_name(0)}")
    else:
        print("ℹ️ PyTorch لا يرى CUDA في هذه الجلسة.")
except ImportError:
    print("ℹ️ PyTorch لم يثبت بعد؛ أعد الفحص بعد خلية تثبيت محرك الصوت.")


In [ ]:
# @title Mount Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

## 1. تثبيت محرك الصوت

شغّل الخلية التالية عند بدء جلسة Colab جديدة. قد يستغرق تنزيل المكتبات والنماذج وقتاً ويحتاج اتصالاً مستقراً.

In [ ]:
# @title Setup runtime environment
from IPython.display import clear_output
import codecs

encoded_url = "uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/"
decoded_url = codecs.decode(encoded_url, "rot_13")

repo_name_encoded = "Nccyvb"
repo_name = codecs.decode(repo_name_encoded, "rot_13")

LOGS_PATH = f"/content/{repo_name}/logs"
BACKUPS_PATH = "/content/drive/MyDrive/abbasdeepBackup"

%cd /content
!git config --global advice.detachedHead false
!git clone {decoded_url}
%cd {repo_name}
clear_output()

!apt update -y
!apt install -y portaudio19-dev

print("Installing requirements...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv venv .venv --python 3.12 --seed
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/{repo_name}/.venv/bin/python
!uv pip install -q ngrok jupyter-ui-poll --system
!uv pip install -q matplotlib-inline websockets --python /content/{repo_name}/.venv/bin/python
!npm install -g -q localtunnel &> /dev/null

!/content/{repo_name}/.venv/bin/python core.py prerequisites --models --pretraineds-hifigan
print("✅ Finished installing requirements!")


## 2. مزامنة الملفات مع Google Drive

تشغّل هذه الخلية مزامنة مجلدات النماذج والتدريب مع مجلد `abbasdeepBackup` في Drive. شغّلها قبل تشغيل الواجهة عند بدء جلسة جديدة.

In [ ]:
# @title Sync with Google Drive
# @markdown 💾 Run this cell to automatically Save/Load models from your mounted drive
# @title
# @markdown This will merge and link your `abbasdeepBackup` folder from gdrive to this notebook
from IPython.display import display, clear_output
from pathlib import Path

non_bak_folders = ["mute", "reference", "zips", "mute_spin", "mute_spin-v2"]
non_bak_path = "/tmp/rvc_logs"


def press_button(button):
  button.disabled = True


def get_date(path: Path):
  from datetime import datetime
  return datetime.fromtimestamp(int(path.stat().st_mtime))


def get_size(path: Path):
  !du -shx --apparent-size "{path}" > /tmp/size.txt
  return open("/tmp/size.txt").readlines().pop(0).split("	")[0] + "B"


def sync_folders(folder: Path, backup: Path):
  from ipywidgets import widgets
  from jupyter_ui_poll import ui_events
  from time import sleep

  local = widgets.VBox([
      widgets.Label(f"Local: {LOGS_PATH.removeprefix('/content/')}/{folder.name}/"),
      widgets.Label(f"Size: {get_size(folder)}"),
      widgets.Label(f"Last modified: {get_date(folder)}")
  ])
  remote = widgets.VBox([
      widgets.Label(f"Remote: {BACKUPS_PATH.removeprefix('/content/')}/{backup.name}/"),
      widgets.Label(f"Size: {get_size(backup)}"),
      widgets.Label(f"Last modified: {get_date(backup)}")
  ])
  separator = widgets.VBox([
      widgets.Label("|||"),
      widgets.Label("|||"),
      widgets.Label("|||")
  ])
  radio = widgets.RadioButtons(
      options=[
          "Save local model to drive",
          "Keep remote model"
      ]
  )
  button = widgets.Button(
      description="Sync",
      icon="upload",
      tooltip="Sync model"
  )
  button.on_click(press_button)

  clear_output()
  print(f"Your local model '{folder.name}' is in conflict with it's copy in Google Drive.")
  print("Please select which one you want to keep:")
  display(widgets.Box([local, separator, remote]))
  display(radio)
  display(button)

  with ui_events() as poll:
    while not button.disabled:
      poll(10)
      sleep(0.1)

  match radio.value:
    case "Save local model to drive":
      !rm -r "{backup}"
      !mv "{folder}" "{backup}"
    case "Keep remote model":
      !rm -r "{folder}"


if Path("/content/drive").is_mount():
  !mkdir -p "{BACKUPS_PATH}"
  !mkdir -p "{non_bak_path}"

  if not Path(LOGS_PATH).is_symlink():
    for folder in non_bak_folders:
      folder = Path(f"{LOGS_PATH}/{folder}")
      backup = Path(f"{BACKUPS_PATH}/{folder.name}")

      !mkdir -p "{folder}"
      !mv "{folder}" "{non_bak_path}" &> /dev/null
      !rm -rf "{folder}"
      folder = Path(f"{non_bak_path}/{folder.name}")
      if backup.exists() and backup.resolve() != folder.resolve():
        !rm -r "{backup}"
      !ln -s "{folder}" "{backup}" &> /dev/null

    for model in Path(LOGS_PATH).iterdir():
      if model.is_dir() and not model.is_symlink():
        backup = Path(f"{BACKUPS_PATH}/{model.name}")

        if model.name == ".ipynb_checkpoints":
          continue

        if backup.exists() and backup.is_dir():
          sync_folders(model, backup)
        else:
          !rm "{backup}"
          !mv "{model}" "{backup}"

    !rm -r "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"

    clear_output()
    print("✅ Models are synced!")

  else:
    !rm "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"
    clear_output()
    print("✅ Models already synced!")

else:
  print("❌ Drive is not mounted, skipping model syncing")
  print("To sync your models, first mount your Google Drive and re-run this cell")

## 3. تخصيص هوية abbasdeep

تفعّل هذه الخلية الترجمة العربية الموجودة في Applio، وتغيّر الاسم والشعار إلى abbasdeep، وتضيف تحسينات RTL وشاشات الهاتف دون استبدال وظائف المحرك.

In [ ]:
# @title إعداد هوية abbasdeep وتحسين العرض للموبايل
from pathlib import Path
import json
import shutil

APPLIO_DIR = Path("/content/Applio")
ASSETS = APPLIO_DIR / "assets"
UI_DIR = ASSETS / "ui"

config_template = ASSETS / "config_template.json"
config_path = ASSETS / "config.json"
if not config_path.exists():
    shutil.copy2(config_template, config_path)
config = json.loads(config_path.read_text(encoding="utf-8"))
config.setdefault("lang", {})
config["lang"]["override"] = True
config["lang"]["selected_lang"] = "ar_AR"
config["discord_presence"] = False
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

logo = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64">
<defs><linearGradient id="a" x1="0" y1="0" x2="1" y2="1"><stop stop-color="#a78bfa"/><stop offset="1" stop-color="#6d4ce8"/></linearGradient></defs>
<rect width="64" height="64" rx="18" fill="url(#a)"/>
<path d="M8 33h6l5-13 8 28 8-35 8 42 7-25 4 13h4" fill="none" stroke="#fff" stroke-width="3.5" stroke-linecap="round" stroke-linejoin="round"/>
</svg>"""
(ASSETS / "applio-logo.svg").write_text(logo, encoding="utf-8")

shell_file = ASSETS / "app_shell.py"
shell_source = shell_file.read_text(encoding="utf-8").replace('<span>Applio</span>', '<span>abbasdeep</span>')
shell_file.write_text(shell_source, encoding="utf-8")

app_file = APPLIO_DIR / "app.py"
app_source = app_file.read_text(encoding="utf-8").replace('title="Applio"', 'title="abbasdeep"')
launch_anchor = "    app, _, _ = Applio.launch(\n"
if launch_anchor not in app_source:
    raise RuntimeError("Applio launch anchor changed upstream; inspect app.py.")
if "        inline=True,\n" not in app_source:
    app_source = app_source.replace(launch_anchor, launch_anchor + "        inline=True,\n", 1)
app_file.write_text(app_source, encoding="utf-8")

css_marker = "/* ABBASDEEP NOTEBOOK UI OVERRIDES v1 */"
css_patch = r"""
/* ABBASDEEP NOTEBOOK UI OVERRIDES v1 */
body, .gradio-container {
  --app-bg: #10101d; --app-surface: #1b1b2d; --app-panel: #161626;
  --app-border: rgba(255,255,255,.10); --app-text: #eceaf7;
  --app-muted: #b5b1ca; --app-heading: #ffffff;
  --button-primary-background-fill: #8b6cf0;
  --button-primary-background-fill-hover: #7658dc;
  --button-primary-text-color: #ffffff; --button-primary-border-color: transparent;
  --color-accent: #a78bfa; --slider-color: #a78bfa;
  direction: rtl !important; text-align: right;
}
#applio-shell { direction: rtl !important; }
#applio-sidebar { inset: 16px 12px 16px auto !important; right:12px !important; left:auto !important; direction:rtl !important; }
#applio-content { margin-right:284px !important; margin-left:0 !important; direction:rtl !important; }
#applio-sidebar button.applio-nav-item { text-align:right !important; }
.applio-nav-item.primary::after { right:4px !important; left:auto !important; }
#applio-navigation { padding-right:0 !important; padding-left:4px !important; }
.applio-page-heading, .applio-card-head { direction:rtl; text-align:right; }
.applio-page .row { direction:rtl; }
.applio-page input:not([type="checkbox"]):not([type="radio"]), .applio-page textarea { text-align:right; }
@media (max-width:800px) {
  #applio-shell { flex-direction:column !important; }
  #applio-sidebar { position:relative !important; inset:auto !important; width:calc(100% - 24px) !important; min-width:0 !important; max-width:none !important; margin:12px 12px 0 !important; padding:12px !important; }
  #applio-menu-toggle { right:12px !important; left:auto !important; top:12px !important; min-width:44px !important; width:44px !important; height:44px !important; }
  #applio-navigation { grid-template-columns:repeat(2,minmax(0,1fr)) !important; direction:rtl !important; }
  #applio-content { margin:0 !important; padding:18px 12px 24px !important; width:100% !important; max-width:100% !important; }
  #applio-sidebar button.applio-nav-item { min-height:44px !important; font-size:12px !important; }
  .applio-page .tab-nav, .applio-page .tab-container[role="tablist"] { max-width:100% !important; overflow-x:auto !important; flex-wrap:nowrap !important; -webkit-overflow-scrolling:touch; }
  .applio-page button { min-height:44px; touch-action:manipulation; }
  .applio-page input:not([type="checkbox"]):not([type="radio"]), .applio-page textarea, .applio-page select { min-height:44px !important; font-size:16px !important; }
  .applio-output-row, .applio-f0-results { grid-template-columns:minmax(0,1fr) !important; }
}
@media (max-width:480px) {
  #applio-content { padding:14px 9px 20px !important; }
  .applio-card { padding:12px !important; }
  .applio-page-heading h1 { font-size:21px !important; }
  .applio-page .row > * { min-width:0 !important; max-width:100% !important; }
  .applio-page audio, .applio-page video, .applio-page img { max-width:100% !important; }
}
@media (prefers-reduced-motion:reduce) { .applio-page button { transition:none !important; } }
"""

css_patch += r"""
/* abbasdeep custom shell styling */
.abbasdeep-brand { padding: 7px 10px 16px !important; gap: 11px !important; }
.abbasdeep-brand > span { font-size: 21px !important; font-weight: 800 !important; letter-spacing: -.5px !important; color: #fff !important; }
.abbasdeep-brand > small { margin-right: auto; color: #c7bdf4 !important; background: rgba(139,108,240,.16) !important; border-color: rgba(167,139,250,.24) !important; }
.abbasdeep-nav-group { margin-bottom: 7px; }
.abbasdeep-nav-heading h2 { font-size: 10px !important; font-weight: 700 !important; letter-spacing: 0 !important; text-transform: none !important; text-align: right !important; }
#applio-sidebar button.applio-nav-item { justify-content: flex-start !important; text-align: right !important; border-radius: 10px !important; min-height: 41px !important; font-size: 12px !important; }
#applio-sidebar button.applio-nav-item.primary { background: rgba(139,108,240,.20) !important; color: #f1edff !important; }
#applio-sidebar button.applio-nav-item.primary::after { right: auto !important; left: 4px !important; background: #a78bfa !important; }
.abbasdeep-sidebar-footer { margin-top: auto; padding: 14px 10px 4px; display: flex; flex-direction: column; gap: 5px; border-top: 1px solid var(--app-border); color: var(--app-muted); font-size: 10px; }
.abbasdeep-sidebar-footer small { color: #8f88a9; font-size: 9px; }
#abbasdeep-topbar { min-height: 54px; margin-bottom: 21px; padding: 0 2px 12px; display: flex; align-items: center; justify-content: space-between; gap: 12px; border-bottom: 1px solid var(--app-border); direction: rtl; }
#abbasdeep-topbar > div { display: flex; align-items: baseline; gap: 10px; flex-wrap: wrap; }
#abbasdeep-topbar strong { color: var(--app-heading); font-size: 15px; font-weight: 800; }
#abbasdeep-topbar span { color: var(--app-muted); font-size: 10px; }
#abbasdeep-topbar .abbasdeep-runtime-pill { padding: 7px 10px; border: 1px solid var(--app-border); border-radius: 30px; background: rgba(139,108,240,.10); color: #c8baff; white-space: nowrap; }
.abbasdeep-page-heading .abbasdeep-kicker { display: block; margin-bottom: 8px; color: #a995ec; font-size: 9px; font-weight: 800; letter-spacing: 1.4px; direction: ltr; text-align: right; }
.abbasdeep-footer { display: flex; justify-content: space-between; gap: 12px; color: #8e89a5; font-size: 10px; }
@media(max-width:800px) {
  .abbasdeep-brand { justify-content: center !important; padding: 0 42px !important; }
  .abbasdeep-brand > small { display: none; }
  .abbasdeep-sidebar-footer { display: none; }
  #abbasdeep-topbar { min-height: 44px; margin-bottom: 14px; }
  #applio-navigation { grid-template-columns: repeat(2,minmax(0,1fr)) !important; }
  #applio-menu-toggle { right: 9px !important; left: auto !important; }
  #applio-sidebar button.applio-nav-item { min-height: 44px !important; padding: 8px 10px !important; font-size: 11px !important; }
}
@media(max-width:480px) {
  #abbasdeep-topbar strong { font-size: 13px; }
  #abbasdeep-topbar span { font-size: 9px; }
  .abbasdeep-footer { flex-direction: column; gap: 4px; }
}
"""

css_file = UI_DIR / "shell.css"
css_source = css_file.read_text(encoding="utf-8")
if css_marker not in css_source:
    css_file.write_text(css_source.rstrip()+"\n\n"+css_patch.strip()+"\n",encoding="utf-8")

# Install the custom abbasdeep navigation shell while reusing Applio's existing tab builders.
custom_shell_source = "from functools import partial\nimport html\nimport gradio as gr\n\nfrom assets.app_shell import (\n    step_card,\n    create_theme as _create_theme,\n    stylesheet as _stylesheet,\n)\n\nSECTIONS = (\n    (\"الصوت\", (\n        (\"inference\", \"تحويل الصوت\"),\n        (\"realtime\", \"الصوت المباشر\"),\n        (\"tts\", \"تحويل النص إلى كلام\"),\n    )),\n    (\"النماذج والتدريب\", (\n        (\"models\", \"مكتبة النماذج\"),\n        (\"training\", \"تدريب نموذج\"),\n        (\"blender\", \"دمج النماذج\"),\n        (\"tensorboard\", \"متابعة التدريب\"),\n    )),\n    (\"الأدوات\", (\n        (\"audio\", \"أدوات الصوت\"),\n        (\"plugins\", \"الإضافات\"),\n    )),\n    (\"النظام\", (\n        (\"terminal\", \"سجل التشغيل\"),\n        (\"settings\", \"الإعدادات\"),\n        (\"report\", \"المساعدة والتقارير\"),\n    )),\n)\n\nDESCRIPTIONS = {\n    \"inference\": \"حوّل ملفاً صوتياً باستخدام نموذج صوت تملكه أو لديك إذن باستخدامه.\",\n    \"realtime\": \"اختبر الصوت المباشر والميكروفون عند توفر دعم المتصفح والجلسة.\",\n    \"tts\": \"أنشئ كلاماً من النص باستخدام النماذج المتاحة لديك.\",\n    \"models\": \"أدر النماذج الصوتية وملفات الفهرسة المحفوظة.\",\n    \"training\": \"جهّز مجموعة بياناتك وابدأ تدريب نموذج صوتي.\",\n    \"blender\": \"ادمج نماذج متوافقة عند الحاجة.\",\n    \"tensorboard\": \"تابع مقاييس التدريب وحالة النموذج.\",\n    \"audio\": \"افحص التسجيلات واستخدم أدوات معالجة الصوت.\",\n    \"plugins\": \"أدر إضافات بيئة التشغيل.\",\n    \"terminal\": \"تابع سجل التشغيل والأخطاء.\",\n    \"settings\": \"اضبط تفضيلات محرك الصوت والأجهزة.\",\n    \"report\": \"راجع معلومات البيئة والمساعدة.\",\n}\n\ndef create_theme():\n    return _create_theme()\n\ndef stylesheet():\n    return _stylesheet()\n\ndef _page_header(key, title, translate):\n    description = DESCRIPTIONS.get(key, \"\")\n    gr.HTML(\n        '<header class=\"applio-page-heading abbasdeep-page-heading\">'\n        '<div><span class=\"abbasdeep-kicker\">ABBASDEEP / WORKSPACE</span>'\n        f'<h1>{html.escape(translate(title))}</h1>'\n        f'<p>{html.escape(translate(description))}</p></div></header>',\n        elem_classes=\"applio-page-header\",\n    )\n\ndef build_workspace(builders, translate=lambda value: value):\n    nav = {}\n    terminal_controls = None\n    entries = [entry for _, group in SECTIONS for entry in group]\n\n    with gr.Row(elem_id=\"applio-shell\", equal_height=False):\n        with gr.Column(scale=0, min_width=220, elem_id=\"applio-sidebar\"):\n            mark = open(\"assets/applio-logo.svg\", encoding=\"utf-8\").read()\n            gr.HTML(\n                f'<div class=\"applio-brand abbasdeep-brand\">{mark}'\n                '<span>abbasdeep</span><small>VOICE STUDIO</small></div>'\n            )\n            menu = gr.Button(\"القائمة\", size=\"sm\", elem_id=\"applio-menu-toggle\")\n            menu.click(\n                fn=None,\n                js=\"\"\"() => {\n                    const sidebar = document.getElementById('applio-sidebar');\n                    const open = sidebar.classList.toggle('mobile-open');\n                    const button = document.getElementById('applio-menu-toggle');\n                    button.setAttribute('aria-expanded', String(open));\n                    button.setAttribute('aria-controls', 'applio-navigation');\n                }\"\"\",\n                queue=False,\n            )\n            with gr.Column(elem_id=\"applio-navigation\"):\n                for section, items in SECTIONS:\n                    with gr.Column(scale=0, min_width=0, elem_classes=\"applio-nav-group\"):\n                        gr.HTML(\n                            f'<h2>{html.escape(section)}</h2>',\n                            elem_classes=\"applio-nav-heading\",\n                        )\n                        for key, label in items:\n                            nav[key] = gr.Button(\n                                label,\n                                variant=\"primary\" if key == \"inference\" else \"secondary\",\n                                elem_classes=[\"applio-nav-item\", f\"applio-nav-{key}\"],\n                            )\n            gr.HTML(\n                '<div class=\"abbasdeep-sidebar-footer\">'\n                '<span>يعمل داخل Google Colab</span>'\n                '<small>abbasdeep · powered by Applio</small>'\n                '</div>'\n            )\n\n        with gr.Column(elem_id=\"applio-content\", min_width=0):\n            gr.HTML(\n                '<div id=\"abbasdeep-topbar\">'\n                '<div><strong>abbasdeep</strong><span>استوديو معالجة الصوت</span></div>'\n                '<span class=\"abbasdeep-runtime-pill\">جلسة Colab</span>'\n                '</div>'\n            )\n            with gr.Tabs(selected=\"inference\", elem_id=\"applio-pages\") as pages:\n                for key, title in entries:\n                    with gr.Tab(title, id=key, elem_id=f\"applio-page-{key}\", elem_classes=\"applio-page\"):\n                        _page_header(key, title, translate)\n                        controls = builders[key]()\n                        if key == \"terminal\":\n                            terminal_controls = controls\n            gr.HTML(\n                '<footer class=\"applio-footer abbasdeep-footer\">'\n                '<span>abbasdeep</span><span>استخدم الأصوات التي تملك حق معالجتها</span>'\n                '</footer>'\n            )\n\n    terminal_timer, terminal_live = terminal_controls or (None, None)\n\n    def navigate(key, live_updates=True):\n        updates = [gr.Tabs(selected=key)]\n        updates.extend(\n            gr.Button(variant=\"primary\" if item == key else \"secondary\")\n            for item in nav\n        )\n        if terminal_timer is not None:\n            updates.append(gr.Timer(active=(key == \"terminal\" and live_updates)))\n        return updates\n\n    for key, button in nav.items():\n        event = button.click(\n            partial(navigate, key),\n            inputs=[terminal_live] if terminal_live is not None else None,\n            outputs=[pages, *nav.values(), *([terminal_timer] if terminal_timer is not None else [])],\n            queue=False,\n            show_progress=\"hidden\",\n        )\n        event.then(\n            fn=None,\n            js=\"\"\"() => requestAnimationFrame(() => {\n                window.dispatchEvent(new Event('resize'));\n                document.getElementById('applio-sidebar')?.classList.remove('mobile-open');\n                document.getElementById('applio-menu-toggle')?.setAttribute('aria-expanded', 'false');\n                window.scrollTo({top: 0, behavior: 'instant'});\n            })\"\"\",\n            queue=False,\n        )\n    return pages\n"
(ASSETS / "abbasdeep_shell.py").write_text(custom_shell_source, encoding="utf-8")

# Switch only the top-level shell import; individual Applio modules remain unchanged.
app_source = app_file.read_text(encoding="utf-8")
app_source = app_source.replace(
    "from assets.app_shell import build_workspace, create_theme, stylesheet",
    "from assets.abbasdeep_shell import build_workspace, create_theme, stylesheet",
)
app_file.write_text(app_source, encoding="utf-8")

print("abbasdeep identity, custom navigation shell and mobile/RTL styles are ready.")


## 4. تشغيل abbasdeep

شغّل الخلية التالية. تُعرض الواجهة داخل مخرجات الدفتر نفسه. لا حاجة لنسخ رابط Gradio أو لصقه في موقع خارجي.

In [ ]:
# @title تشغيل واجهة abbasdeep داخل دفتر Colab
# يشغّل التطبيق باستخدام نفس البيئة الافتراضية التي ثُبّتت فيها مكتبات Applio.
# ستظهر الواجهة في مخرجات هذه الخلية؛ لا تغلق الجلسة أثناء الاستخدام.
from pathlib import Path
import subprocess
import sys

APPLIO_DIR = Path("/content/Applio")
VENV_PYTHON = APPLIO_DIR / ".venv" / "bin" / "python"
APP_FILE = APPLIO_DIR / "app.py"

if not APP_FILE.is_file():
    raise FileNotFoundError(
        f"لم يتم العثور على app.py داخل {APPLIO_DIR}. "
        "أعد تشغيل خلية تثبيت محرك الصوت."
    )

if not VENV_PYTHON.is_file():
    raise FileNotFoundError(
        f"لم يتم العثور على Python الخاص ببيئة Applio: {VENV_PYTHON}. "
        "أعد تشغيل خلية تثبيت محرك الصوت لإنشاء .venv."
    )

print(f"Python runtime: {VENV_PYTHON}")
print("Starting abbasdeep with Applio's installed virtual environment...")

result = subprocess.run(
    [
        str(VENV_PYTHON),
        str(APP_FILE),
        "--listen",
        "--share",
        "--client",
    ],
    cwd=str(APPLIO_DIR),
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        f"توقف تطبيق abbasdeep برمز خروج {result.returncode}. "
        "راجع سجل الخلية أعلاه لمعرفة الحزمة أو الإعداد الذي تعذر تشغيله."
    )
